# 04 · 原片精改 v2 专业高清管线（IndexTTS-2 播音级清晰中文克隆 + FaceFusion HyperSwap-256 / EDTalk-256 / GPEN-512 / XSeg 遮罩）

- **升级亮点**：
  1. **台词清晰不模糊**：由 Bilibili **IndexTTS-2 / 2.5**（原生中文字符+拼音混合建模 + BigVGAN2 声码器）取代 Chatterbox，保持 `1.00x` 原速清晰吐字，并在说话段将原片 Demucs 残声压低 `-18dB`，彻底消除重影与水下感。
  2. **人物高清稳定不抖**：采用官方 **FaceFusion v3** `hyperswap_1b_256`（4 倍像素）+ `edtalk_256`（256px 精准口型，告别 96px Wav2Lip 糊嘴）+ `gpen_bfr_512`（标准 5 点 FFHQ 仿射对齐还原）+ `xseg_1` 遮罩（100% 保留官帽帽檐与胡须）。
  3. **先做 15 秒验证**：默认 `CLIP_END_SEC = 17.2`（完整覆盖特朗普 `shot05c` + 鲁比奥 `shot06` 两句完整对话），3–5 分钟出样片确认效果后再跑全片 `250.7s`！

In [ ]:
#@title 1. 挂载 5TB Google Drive、拉取最新仓库并构建双独立 GPU 环境（IndexTTS-2 + FaceFusion v3）
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}
PROJECT_ID = "02_yongzheng_ep12_huang" #@param {type:"string"}

import os, shutil, subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def run_stream(cmd, cwd=None, extra_env=None):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")

if not Path(REPO_DIR).exists():
    run_stream(["git", "clone", REPO_URL, REPO_DIR])
else:
    run_stream(["git", "-C", REPO_DIR, "pull", "--ff-only"])

PROJECT_DIR = Path(f"/content/drive/MyDrive/AI_Video/projects/{PROJECT_ID}")
shutil.copy2(f"{REPO_DIR}/projects/{PROJECT_ID}/dubbing_plan_shot01_21.json", PROJECT_DIR / "dubbing_plan_shot01_21.json")

run_stream(["nvidia-smi"])
run_stream(["pip", "install", "-q", "-U", "uv", "huggingface_hub"])

# 环境 A：IndexTTS-2（原生 uv 隔离环境 + 5TB Drive 权重缓存）
INDEXTTS_DIR = Path("/content/index-tts")
if not INDEXTTS_DIR.exists():
    run_stream(["git", "clone", "--depth", "1", "https://github.com/index-tts/index-tts.git", str(INDEXTTS_DIR)])
run_stream(["uv", "sync", "--no-dev"], cwd=str(INDEXTTS_DIR))
run_stream(["uv", "pip", "install", "--python", str(INDEXTTS_DIR / ".venv/bin/python"), "demucs", "soundfile"])

# 缓存 IndexTTS-2 权重到 5TB Drive（若已存在直接秒复用）
CKPT_DIR = Path("/content/drive/MyDrive/AI_Video/shared/models/indextts2")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if not (CKPT_DIR / "config.yaml").exists():
    print("[下载权重] IndexTeam/IndexTTS-2 -> 5TB Google Drive 共享缓存...", flush=True)
    run_stream([
        "python3", "-c",
        f"from huggingface_hub import snapshot_download; snapshot_download('IndexTeam/IndexTTS-2', local_dir='{CKPT_DIR}')"
    ])

# 环境 B：官方 FaceFusion v3（独立 Python 3.10 CUDA 环境 + 5TB Drive 模型缓存，内置 HyperSwap-256 + EDTalk-256 + GPEN-512 + XSeg）
FF_DIR = Path("/content/facefusion")
if not FF_DIR.exists():
    run_stream(["git", "clone", "--depth", "1", "https://github.com/facefusion/facefusion.git", str(FF_DIR)])
FF_CACHE = Path("/content/drive/MyDrive/AI_Video/shared/models/facefusion_assets")
FF_CACHE.mkdir(parents=True, exist_ok=True)
ff_assets_models = FF_DIR / ".assets" / "models"
ff_assets_models.parent.mkdir(parents=True, exist_ok=True)
if not ff_assets_models.exists():
    os.symlink(str(FF_CACHE), str(ff_assets_models))
FF_VENV = Path("/content/venv_ff")
if not FF_VENV.exists():
    run_stream(["uv", "venv", "--python", "3.10", str(FF_VENV)])
FF_PY = str(FF_VENV / "bin/python")
run_stream(["uv", "pip", "install", "--python", FF_PY, "-r", str(FF_DIR / "requirements.txt")])
run_stream(["uv", "pip", "install", "--python", FF_PY, "--force-reinstall", "onnxruntime-gpu", "nvidia-cudnn-cu12", "nvidia-cublas-cu12", "nvidia-cuda-runtime-cu12", "insightface", "pillow"])

print("\n[v2 双环境就绪] IndexTTS-2 与 FaceFusion v3 (CUDA) 安装完成！")

In [ ]:
#@title 2. Stage 1：精确裁剪 + IndexTTS-2 播音级清晰中文克隆（1.00x 原速，零重影）
WORK_DIR = "/content/work_replace_v2" #@param {type:"string"}
CLIP_END_SEC = 17.2 #@param {type:"number"}

import glob, os
from IPython.display import Audio, display

TTS_PY = "/content/index-tts/.venv/bin/python"
nv_libs = glob.glob("/content/index-tts/.venv/lib/python*/site-packages/nvidia/*/lib")
run_stream([
    TTS_PY, "-u", f"{REPO_DIR}/scripts/video_replace_v2.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "dub",
    "--clip-end-sec", str(CLIP_END_SEC),
    "--indextts-dir", "/content/index-tts",
    "--indextts-ckpt-dir", str(CKPT_DIR),
], extra_env={"LD_LIBRARY_PATH": ":".join(nv_libs + [os.environ.get("LD_LIBRARY_PATH", "")])})

for wav in sorted(Path(WORK_DIR, "tts_v2").glob("*_clean.wav")):
    print(f"试听清晰中文克隆: {wav.name}")
    display(Audio(str(wav)))

In [ ]:
#@title 3. Stage 2：FaceFusion v3 高清换脸 (HyperSwap-256) + 精准口型 (EDTalk-256) + FFHQ 修复 (GPEN-512) + 自动释放 GPU
OUTPUT_NAME = "02_yongzheng_ep12_huang_v2_15s_preview.mp4" #@param {type:"string"}
FACE_SWAPPER = "hyperswap_1b_256" #@param ["hyperswap_1b_256", "hyperswap_1a_256", "inswapper_128_fp16"]
FACE_ENHANCER = "gpen_bfr_512" #@param ["gpen_bfr_512", "codeformer", "gfpgan_1.4"]
LIP_SYNCER = "edtalk_256" #@param ["edtalk_256", "wav2lip_gan_96"]
SKIP_LIPSYNC = False #@param {type:"boolean"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import glob, os, subprocess, time
from pathlib import Path

ff_nv_libs = glob.glob("/content/venv_ff/lib/python*/site-packages/nvidia/*/lib") + ["/usr/local/cuda/lib64", "/usr/lib/x86_64-linux-gnu"]
cmd = [
    FF_PY, "-u", f"{REPO_DIR}/scripts/video_replace_v2.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "swap",
    "--facefusion-dir", "/content/facefusion",
    "--output-name", OUTPUT_NAME,
    "--face-swapper-model", FACE_SWAPPER,
    "--face-enhancer-model", FACE_ENHANCER,
    "--lip-syncer-model", LIP_SYNCER,
]
if SKIP_LIPSYNC:
    cmd.append("--skip-lipsync")

run_stream(cmd, extra_env={"LD_LIBRARY_PATH": ":".join(ff_nv_libs + [os.environ.get("LD_LIBRARY_PATH", "")])})

subprocess.run(["sync"], check=False)
time.sleep(10)

final_mp4 = PROJECT_DIR / "outputs" / "final_cut" / OUTPUT_NAME
print(f"\n=== v2 高清验证样片完成！位置: {final_mp4} ===")

if AUTO_DISCONNECT and final_mp4.exists() and final_mp4.stat().st_size > 500_000:
    print("[自动止损] 成片已安全落盘 5TB Google Drive，自动释放 GPU 运行时！")
    from google.colab import runtime
    runtime.unassign()